# Using PySCF with GPU4PySCF in Hybrid CPU-GPU Workflows
3rd PySCF Workshop, University of Chicago, 2026

## Outline

* Installation
* Running GPU4PySCF
* Multi-GPU Support
* Nvidia cuEST Interface
* Integrating GPU4PySCF into a PySCF Codebase

## Available Features and Expected Performance                                              
GPU4PySCF can provide substantial performance improvements over CPU calculations.         
- A very rough estimate:                                                         
  **1 Nvidia A100 GPU ≈ 500 CPU cores** 

| Features | Supported sizes | Speedup vs. PySCF |
|---|---|---:|
| Density fitting DFT (energy, gradient, Hessian) |3000 bfns | > 1,000 |
| Integral-direct DFT (energy, gradient, Hessian) | 30000 bfns | 500–1,000 |
| TDDFT (energy, gradient, non-adiabatic coupling) || 500-1,000 |
| PCM solvent model (energy, gradient, Hessian) |20000 bfns | > 1,000 |
| DFT with periodic boundary conditions (PBC) |10000 bfns | 500-1,000 |

---

## Installation

### Option 1: Install Precompiled Wheels

Precompiled releases are provided for three CUDA SDK generations:

- CUDA 11: Generally lower performance.
- CUDA 12: ECP calculations are relatively slow in this release.
- CUDA 13: Expected to provide the best performance.

For example,

```bash
pip install gpu4pyscf-cuda13x
```

#### cuTENSOR Is Strongly Recommended

```bash
pip install cutensor-cu12==2.2.0
```

- `cutensor` is **not a required dependency**, but installing it is strongly recommended.
- GPU4PySCF contains optimizations specifically designed to take advantage of cuTensor.

Which cuTENSOR version?
- The latest version is **not always the best choice**, particularly for cuTENSOR.
- We have encountered issues with some of the latest cuTENSOR releases.

#### GPU Compute Compatibility
The precompiled wheels do **not** contain binaries for every compute capability (`sm_*`).

Currently, the following architectures are precompiled:
- `sm70` : V100
- `sm80` (along with PTX virtual code) : A100, A30, A800
- `sm90` : H100, H200
- `sm120` : B100, B200

If your GPU requires a different compute capability:
- CUDA SDK may trigger **JIT compilation**.
- It may translate the `sm80` PTX code to the architecture required by your GPU. This trans-compilation can take a ~1 hour.


## Option 2: Compile from Source

```bash
pip install gpu4pyscf
```

Alternatively:
- Clone the GPU4PySCF repository from GitHub.
- Build it in a way similar to building PySCF from source.

## Option 3: No Local GPUs — Try `volcqc-qcclient`

- `volcqc-qcclient` is a product provided by ByteDance Volcano Engine.
- It only supports a subset of GPU4PySCF functionality.

---

## Running GPU4PySCF

### Directly Importing GPU4PySCF

A straightforward way to use GPU4PySCF is:

```python
from gpu4pyscf import xxx
```

GPU4PySCF generally follows the package structure and naming conventions of PySCF.
However, the correspondence is **not always one-to-one**.

### Using `to_gpu()`

The `to_gpu()` conversion provides a simpler entry point for GPU4PySCF, which allows you

1. Initialize and configure the calculation using the familiar PySCF API.
2. Call `to_gpu()` to transfer the PySCF objects and configurations to the corresponding GPU4PySCF object.
3. Use `to_cpu()` to transfer the GPU4PySCF objects back to the PySCF objects.

A few caveats:

- Many PySCF method classes provide `to_gpu()`, but **not every class does**.
- Both PySCF and GPU4PySCF continue to evolve, the behavior of `to_gpu()` can differ slightly between versions.
- Importing GPU4PySCF can modify PySCF `to_gpu()` behavior (check the `gpu4pyscf/_patch_pyscf.py` file).

To support the newly developed features, some PySCF `to_gpu()` functions are overwritten upon importing GPU4PySCF.

If unsure, **`import gpu4pyscf` first, let it patch PySCF `to_gpu()` methods**

#### When Can I Use `to_gpu()`?
> Recommendation: call the `to_gpu()` conversion early if possible.

In [ ]:
import gpu4pyscf # import gpu4pyscf first, allowing it patching pyscf
import pyscf

mol = pyscf.M(atom=gpu4pyscf_src/benchmarks/molecules/organic/020_Vitamin_C.xyz', basis='def2-tzvp')

mf = mol.to_gpu().RKS(xc='wb97mv').density_fit(auxbasis='autoaux').PCM().run() # recommended

mf = mol.RKS(xc='wb97mv').to_gpu().density_fit(auxbasis='autoaux').PCM().run()

mf = mol.RKS(xc='wb97mv').density_fit(auxbasis='autoaux').to_gpu().PCM().run()

mf = mol.RKS(xc='wb97mv').density_fit(auxbasis='autoaux').PCM().to_gpu().run()

mol.RKS(xc='wb97mv').density_fit(auxbasis='autoaux').PCM().Gradients().to_gpu().run()

mf = mf.to_cpu()

---

### Examples

#### A PBC single point calculation

In [ ]:
from pyscf.pbc.gto import Cell
cell = Cell()
cell.fromfile('gpu4pyscf_src/benchmarks/crystals/MgO_primitive.cif')
cell.basis = 'gth-dzvp'
cell.pseudo = 'gth-pbe'
cell.build()
# k-point sampling calculation
kmesh = [5,5,5]
mf = cell.to_gpu().KRKS(xc='pbe', kpts=cell.make_kpts(kmesh))
# For calculations with pseudo potential, Multigrid algorithm can be enabled,
# which is 2 - 3 orders of maginitude faster than the default setting.
mf = mf.multigrid_numint().run()

#### Geometry and lattice optimization

In [ ]:
# The optimizer is a wrapper for the geometry optimization algorithms provided by ASE package.

optimizer = mf.Gradients().optimizer()

# By default: optimize both atomic positions and lattice parameters
optimizer.run()

# Optimize atomic positions only
optimizer.target = 'atoms'
optimizer.run()

# Optimize lattice parameters only
optimizer.target = 'lattice'
optimizer.run()

---

## Multi-GPU Support

Multi-GPU scaling depends strongly on the computational method.

### Direct SCF

Direct SCF generally shows **good multi-GPU scaling**:

- **K matrix:** scales well.
- **XC matrix:** scales well.
- Good multi-GPU speedup can be expected for calculations:
  - SCF energy
  - Nuclear gradients
  - Hessians
  - TDDFT
  - TDDFT derivative couplings

### Density Fitting

SCF calculations can benefit from multiple GPUs

- DF tensor initialization: scales poorly with multiple GPUs. Typically accounts for **< 10%** of the total cost.
- DF tensor storage: the three-index DF tensor is distributed across GPUs.
- K matrix contraction: shows strong multi-GPU scaling.
- For small systems (< 1,000 basis functions): Multi-GPU execution may provide little benefit. It can even be slower due to communication and synchronization overhead.

Multi-GPU benefits are more limited for derivative calculations:

- Nuclear gradients: single GPU only.
- TDDFT: single GPU only.
- TDDFT derivative couplings: single GPU only.
- **Nuclear Hessians:** support multiple GPUs, but with significant overhead.
  - Multi-GPU execution becomes beneficial for at least 70 atoms.

## Enabling cuEST (CUDA Electronic Structure Theory)

- cuEST is designed to accelerate **ERI computations** for quantum chemistry applications on NVIDIA GPUs.
- cuEST can provide substantial performance improvements over GPU4PySCF density-fitting J/K construction.

Approximate speedup relative to the native GPU4PySCF density-fitting J/K implementation:

| GPU Architecture | Approximate Speedup |
|---|---:|
| Ampere series | ~1.5× |
| Hopper series | ~3× |
| Blackwell series | ~5× |
| RTX PRO 6000 | ~15× |

### Limitations

- DFT XC integrals and density evaluation are inefficient. 
- Not support Volta GPUs
- Not support Python 3.10 or earlier

### Using cuEST with GPU4PySCF

cuEST can be accessed through the interface provided by GPU4PySCF.

- The GPU4PySCF interface automatically handles differences between the two packages:
  - Atomic orbital ordering conventions.
  - DFT grid schemes.

In [ ]:
from gpu4pyscf.lib.cuest_wrapper import apply_cuest_wrapper
mf = mol.to_gpu().RKS(xc='pbe0').density_fit()
mf = apply_cuest_wrapper(mf)

# Turn off cuEST DFT XC part
mf._numint.turn_on_cuest_xc = False

---

## Integrating GPU4PySCF into a PySCF Codebase

### Data Residence

Tensors and arrays may reside in either CPU memory or GPU memory. An ideal convention would be:

- NumPy input -> NumPy output
- CuPy input -> CuPy output

However, only a limited number of functions in GPU4PySCF currently follow this convention.

* Input Arrays
  - **CuPy arrays are preferred** for GPU4PySCF functions.
  - NumPy arrays are usually accepted and handled correctly.

* Calculation Results (as attributes of a class), typically
  - Energies (energy, nuclear gradients, nuclear Hessians) -> NumPy array.
  - Wavefunction (orbital coefficients, orbital energies, orbital occupancies, excitation amplitudes) -> CuPy array.

When calling GPU4PySCF functions from PySCF code, explicitly handle NumPy/CuPy conversions when needed:

In [ ]:
import cupy as cp

# Convert to a CuPy array
cp.asarray(x)

# Convert to a NumPy array
cp.asnumpy(x)

### J/K Matrices for Molecular Systems

- The `gpu4pyscf.scf.hf` module provides a `get_jk` function that supports density matrices of arbitrary shape. Its API is the same to its PySCF counterpart.
- Multi-GPU computation is supported.
- The `gpu4pyscf.scf.jk` module works differently from `pyscf.scf.jk`.

In [ ]:
from gpu4pyscf.scf.hf import get_jk
vj, vk = get_jk(mol, dm)

Repeated `get_jk` calculations:
- Initialize the J/K handler once.
- Reuse the handler whenever possible to avoid repeated initialization overhead.

In [ ]:
from gpu4pyscf.scf.jk import _VHFOpt
vhfopt = _VHFOpt(mol)
while True:
    vj, vk = get_jk(mol, dm, vhfopt=vhfopt)

`get_j()` and `get_k()` vs. `get_jk()`

It is generally **more efficient to evaluate J and K separately** than computing both within a single `get_jk()` kernel. `get_j()` and `get_k()` use **different algorithms**, each is optimized.

In [ ]:
from gpu4pyscf.scf.hf import SCF
vj = SCF(mol).get_j(mol, dm)
vk = SCF(mol).get_k(mol, dm)

### J/K Matrices for Extended Systems

#### Access Coulomb matrix

Available algorithms to compute Coulomb matrices: Multigrid, GDF, RSJK

* Performance: Multigrid >> GDF > RSJK
* Accuracy: Multigrid = RSJK > GDF

In [ ]:
# via GDF
from gpu4pyscf.pbc.df import GDF
J = GDF(cell).get_j(dm, kpts=kpts)

# via RSJK
from gpu4pyscf.pbc.scf.rsjk import get_j
J = get_j(cell, dm, kpts=kpts)

# via MultigridNumInt
from gpu4pyscf.pbc.dft.multigrid_v2 import MultiGridNumInt
J = MultiGridNumInt(cell).get_j(dm, kpts=kpts)

#### Access exact exchange matrix

* Performance: GDF > RSJK >> AFTDF ~= FFTDF
* Accuracy: FFTDF = AFTDF ~= RSJK > GDF

In [ ]:
# via GDF
from gpu4pyscf.pbc.df import GDF
K = GDF(cell).get_k(dm, kpts=kpts, exxdiv='ewald')

# via RSJK
from gpu4pyscf.pbc.scf.rsjk import get_k
K = get_k(cell, dm, kpts=kpts, exxdiv='ewald')

### Four-Index Integrals

GPU4PySCF currently does **not** provide a stable, general-purpose API for four-index integrals.

A module for computing four-index integrals is available, but:
- The current implementation is **not memory efficient**.
- The API is still unstable and **subject to rapid changes**.

> **Recommendation:** Use density fitting whenever possible.

### Accessing Density-Fitting Tensors for Molecular Systems

The tensor can be accessed internally through `mf.with_df._cderi`. However, its storage format is **not compatible** with the PySCF `mf.with_df._cderi`.

Key differences:

- **Compressed storage:** Orbital pairs with negligible overlap are discarded.
- **Multi-GPU distribution:** The DF tensor is distributed as if there are multiple GPUs.

> **Recommended API: `with_df.loop()`**, insteand of accessing `mf.with_df._cderi` directly directly.

When running on a single GPU:

In [ ]:
for cderi_Pij, _ in with_df.loop():
    eri += cp.einsum('Pij,Pkl->ijkl', cderi_Pij, cderi_Pij)

#### DF tensor with Multiple GPUs

A basic implementation looks like:

In [ ]:
eri = [] 
for device_id in range(num_devices):
    eri_on_device = 0
    with cp.cuda.Device(device_id):
        for cderi, _ in with_df.loop():
            # Process the contraction on each device
            eri_on_device += cp.einsum('Pij,Pkl->ijkl', cderi, cderi)
    eri.append(eri_on_device)

# Move results from worker GPUs to device 0
eri = [cp.asarray(x) for x in eri]

# Reduce on device 0
eri = sum(eri)

#### Simplifying Multi-GPU Operations

GPU4PySCF provides helper functions in `gpu4pyscf.lib.multi_gpu` for common multi-GPU operations:

In [ ]:
from gpu4pyscf.lib import multi_gpu

def proc():
    eri = 0
    for cderi, _ in with_df.loop():
        eri += cp.einsum('Pij,Pkl->ijkl', cderi, cderi)
    return eri

eri = multi_gpu.arrays_reduce(
    multi_gpu.run(proc, non_blocking=True)
)

#### MO Integrals

In the upcoming **GPU4PySCF 1.9 release**, the density-fitting class will provide an `ao2mo()` API to simplify access to MO-transformed integrals.

In [ ]:
o_orb = mf.mo_coeff[:,:nocc]
v_orb = mf.mo_coeff[:,nocc:]
mf.with_df.ao2mo([o_orb, o_orb, v_orb, v_orb])

### Accessing Density-Fitting Tensors for PBC Systems

The DF tensor is held by `with_df._cderi`. Its storage format is **not compatible** with the PySCF counterpart.

- `with_df._cderi` is stored in **host memory, not distributed across multiple GPUs**.

#### Gamma Point

For PBC calculations at the gamma point, the recommended API is `with_df.loop_gamma_point()`

> PBC GDF class also provides `with_df.loop()`. However, it is designed primarily for **k-point sampling**.


In [ ]:
eri = 0
batch_size = 200
for cderi, _, sign in with_df.loop_gamma_point(batch_size):
    eri += sign * cp.einsum('Pij,Pkl->ijkl', cderi, cderi)

#### Density-Fitting Tensors for PBC k-Point Sampling

The PBC implementation uses several optimizations to reduce memory and computational cost.

1. Time-reversal symmetry
   
   $T_{P,\mu\nu}^{\mathbf{k}_P,\mathbf{k}_\mu,\mathbf{k}_\nu} = (T_{P,\mu\nu}^{-\mathbf{k}_P,-\mathbf{k}_\mu,-\mathbf{k}_\nu})^*$

2. Permutation symmetry between two orbitals
   
   $T_{P,\mu\nu}^{-\mathbf{k}_P,-\mathbf{k}_\mu,-\mathbf{k}_\nu} = T_{P,\nu\mu}^{-\mathbf{k}_P,\mathbf{k}_\nu,\mathbf{k}_\mu}$

3. Sparsity in supercell AO representation.

   Performing a Wannier-like back-transformation to get real-space supercell representation:
   
   $T_{P,\mu(\mathbf{R})\nu(\mathbf{S})}^{\mathbf{k}_P} = T_{P,\mu\nu}^{\mathbf{k}_P,\mathbf{k}_\mu,\mathbf{k}_\nu} e^{i \mathbf{k}_\mu\cdot\mathbf{R}} e^{-i \mathbf{k}_\nu\cdot\mathbf{S}}$

   In the supercell representation, more orbital pairs $\mu(\mathbf{R})\nu(\mathbf{S})$ are negligble.

4. Permutation for real orbitals in supercell.

   $T_{P,\mu(\mathbf{R})\nu(\mathbf{S})}^{\mathbf{k}_P} = T_{P,\nu(\mathbf{S})\mu(\mathbf{R})}^{\mathbf{k}_P}$

The `with_df.loop()` API:

* The `with_df.loop()` API performs most of the required decompression and representation transformations automatically (condition 1, 3, 4). However, permutation symmetry (condition 2) must be handled manually by the caller.

* `with_df.loop()` does **not** iterate over individual k-points one at a time. In each iteration, it returns a **batch of `cderi` tensors**. Each tensor in the list corresponds to a particular k-point.

In [ ]:
import cupy as cp
from gpu4pyscf.pbc.df import GDF
from gpu4pyscf.pbc.lib.kpts_helper import kk_adapted_iter

with_df = GDF(cell, kpts)

k_adapt_dic = {}                                                                        
for kP, kP_conj, ki_idx, kj_idx in kk_adapted_iter(with_df.kmesh):                         
    k_adapt_dic[kP] = kP_conj, ki_idx, kj_idx                                                   

eri = cp.zeros((nk, nk, nk, nao, nao, nao, nao), dtype=cp.complex128)
for kP, Lij, sign in with_df.loop(blksize):
    kP_conj, ki_idx, kj_idx = k_adapt_dic[kP]
    eri[ki_idx,kj_idx,:, :,:,:,:] += cp.einsum(
        'PLij,QLkl->PQijkl', Lij[:,:,ki_idx,kj_idx], Lij[:,:,kj_idx,ki_idx].conj())